# Notebook 04 — Crowd Flow And Dispatch Intelligence



**Goal:** Convert model outputs into event-safety intelligence for commanders.



This notebook focuses on what the event team actually needs:



1. Zone-wise crowd density estimates

2. Short-term crowd flow and congestion change signals

3. Bottleneck risk ranking

4. Visual anomaly evidence

5. Natural-language situational summary

6. Suggested responder dispatch order



> This notebook does not do person identification. It works with zone-level and incident-level intelligence only.

In [ ]:
# Auto-detect repo root for JupyterLab / GCP / local execution

from pathlib import Path

import sys

import json

import math



def find_repo_root(start=None):

    start_path = Path(start or Path.cwd()).resolve()

    for candidate in [start_path, *start_path.parents]:

        if (candidate / 'src').exists() and (candidate / 'notebooks').exists():

            return candidate

    return start_path



REPO_ROOT = find_repo_root()

DATA_ROOT = REPO_ROOT / 'data'

CKPT_ROOT = REPO_ROOT / 'checkpoints'

EXPS_ROOT = REPO_ROOT / 'experiments'

EXPS_ROOT.mkdir(exist_ok=True)

sys.path.insert(0, str(REPO_ROOT))



import numpy as np

import pandas as pd

import torch

import matplotlib.pyplot as plt

from PIL import Image



from src.data_loaders import get_shanghaitech_loaders, get_ucsd_loaders, load_metr_la

from src.models import AdaptiveCSRNet, ConvAE, FutureFrameNet, AdaptiveNASGNN, GCNGRU



DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Device: {DEVICE}')



def load_checkpoint_if_exists(model, ckpt_path):

    ckpt_path = Path(ckpt_path)

    if not ckpt_path.exists():

        return False

    ckpt = torch.load(ckpt_path, map_location=DEVICE)

    state = ckpt.get('model', ckpt)

    model.load_state_dict(state, strict=False)

    model.eval()

    return True



def zone_grid(height, width, rows=3, cols=3):

    h_edges = np.linspace(0, height, rows + 1, dtype=int)

    w_edges = np.linspace(0, width, cols + 1, dtype=int)

    zones = []

    for r in range(rows):

        for c in range(cols):

            zones.append({

                'name': f'Zone-{chr(65 + r * cols + c)}',

                'r0': h_edges[r], 'r1': h_edges[r + 1],

                'c0': w_edges[c], 'c1': w_edges[c + 1],

                'row': r, 'col': c,

            })

    return zones



def softmax_np(x):

    x = np.asarray(x, dtype=np.float64)

    e = np.exp(x - x.max())

    return e / (e.sum() + 1e-8)

In [ ]:
# Configuration

CFG = {

    'density_part': 'A',

    'target_size': (576, 768),

    'batch_size': 1,

    'zone_rows': 3,

    'zone_cols': 3,

    'forecast_seq_in': 12,

    'forecast_seq_out': 12,

    'dispatch_units': [

        {'unit_id': 'MED-1', 'type': 'medical',  'x': 0.15, 'y': 0.20},

        {'unit_id': 'MED-2', 'type': 'medical',  'x': 0.80, 'y': 0.15},

        {'unit_id': 'SEC-1', 'type': 'security', 'x': 0.20, 'y': 0.85},

        {'unit_id': 'SEC-2', 'type': 'security', 'x': 0.85, 'y': 0.82},

    ],

}



if DEVICE == 'cpu':

    CFG['target_size'] = (288, 384)



print(json.dumps(CFG, indent=2))

In [ ]:
# Load trained models if available

density_model = AdaptiveCSRNet(load_weights=False).to(DEVICE)

density_loaded = load_checkpoint_if_exists(

    density_model,

    CKPT_ROOT / f'adaptive_csrnet_sha{CFG["density_part"]}' / 'best.pt'

)



anomaly_model = ConvAE(in_channels=1, base_ch=32).to(DEVICE)

anomaly_loaded = load_checkpoint_if_exists(

    anomaly_model,

    CKPT_ROOT / 'convae_ped2' / 'best.pt'

)



forecast_model = None

forecast_status = 'heuristic'

metr_h5 = DATA_ROOT / 'metr-la' / 'metr-la.h5'

metr_adj = DATA_ROOT / 'metr-la' / 'adj_mx.pkl'

if metr_h5.exists():

    _, _, metr_test, scaler, adj, num_nodes = load_metr_la(

        h5_path=str(metr_h5),

        adj_path=str(metr_adj) if metr_adj.exists() else None,

        seq_in=CFG['forecast_seq_in'],

        seq_out=CFG['forecast_seq_out'],

        batch_size=16,

    )

    candidate = AdaptiveNASGNN(

        num_nodes=num_nodes,

        in_features=2,

        hidden_dim=64,

        num_layers=2,

        seq_out=CFG['forecast_seq_out'],

    ).to(DEVICE)

    if load_checkpoint_if_exists(candidate, CKPT_ROOT / 'nas_gnn_retrain' / 'best.pt'):

        forecast_model = candidate

        forecast_status = 'nas_gnn'

    else:

        candidate = GCNGRU(

            num_nodes=num_nodes,

            in_features=2,

            hidden_dim=64,

            num_layers=2,

            seq_out=CFG['forecast_seq_out'],

        ).to(DEVICE)

        if load_checkpoint_if_exists(candidate, CKPT_ROOT / 'gcn_gru_metrla' / 'best.pt'):

            forecast_model = candidate

            forecast_status = 'gcn_gru'



print({

    'density_checkpoint': density_loaded,

    'anomaly_checkpoint': anomaly_loaded,

    'forecast_mode': forecast_status,

})

In [ ]:
# Zone-wise density estimation from a crowd image

density_train, density_test = get_shanghaitech_loaders(

    data_root=str(DATA_ROOT),

    part=CFG['density_part'],

    batch_size=1,

    target_size=CFG['target_size'],

    num_workers=0,

)



imgs, density_gt, counts_gt = next(iter(density_test))

img = imgs.to(DEVICE)



with torch.no_grad():

    if density_loaded:

        density_pred = density_model(img).detach().cpu().numpy()[0, 0]

    else:

        density_pred = density_gt.numpy()[0, 0]



density_img = density_gt.numpy()[0, 0]

img_np = imgs[0].permute(1, 2, 0).numpy()

img_np = np.clip(img_np * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406]), 0, 1)



zones = zone_grid(density_pred.shape[0], density_pred.shape[1], CFG['zone_rows'], CFG['zone_cols'])

zone_density = []

for zone in zones:

    patch = density_pred[zone['r0']:zone['r1'], zone['c0']:zone['c1']]

    zone_density.append({

        'zone': zone['name'],

        'density_sum': float(patch.sum()),

        'density_mean': float(patch.mean()),

        'row': zone['row'],

        'col': zone['col'],

    })



zone_density_df = pd.DataFrame(zone_density).sort_values('density_sum', ascending=False).reset_index(drop=True)

zone_density_df

In [ ]:
# Estimate short-term crowd flow from UCSD clip differences

ucsd_train, ucsd_test = get_ucsd_loaders(

    data_root=str(DATA_ROOT),

    ped='ped2',

    img_size=(CFG['target_size'][0], CFG['target_size'][1]),

    batch_size=1,

    clip_len=8,

    num_workers=0,

)



batch = next(iter(ucsd_test))

if isinstance(batch, (list, tuple)):

    clips = batch[0]

else:

    clips = batch



clip = clips[0].numpy()  # [T, C, H, W]

if clip.shape[1] == 1:

    clip_gray = clip[:, 0]

else:

    clip_gray = clip.mean(axis=1)



flow_energy = np.abs(np.diff(clip_gray, axis=0))

flow_map = flow_energy.mean(axis=0)



flow_zones = []

for zone in zone_grid(flow_map.shape[0], flow_map.shape[1], CFG['zone_rows'], CFG['zone_cols']):

    patch = flow_map[zone['r0']:zone['r1'], zone['c0']:zone['c1']]

    flow_zones.append({

        'zone': zone['name'],

        'flow_intensity': float(patch.mean()),

        'flow_sum': float(patch.sum()),

    })



flow_df = pd.DataFrame(flow_zones).sort_values('flow_intensity', ascending=False).reset_index(drop=True)

flow_df

In [ ]:
# Forecast congestion trend and derive bottleneck proxy by pseudo-zone

forecast_zone_df = pd.DataFrame(columns=['zone', 'current_load', 'forecast_load', 'trend', 'congestion_risk'])



if metr_h5.exists():

    batch_x, batch_y = next(iter(metr_test))

    batch_x_dev = batch_x.to(DEVICE)

    adj_dev = adj.to(DEVICE) if isinstance(adj, torch.Tensor) else torch.tensor(adj, dtype=torch.float32, device=DEVICE)



    with torch.no_grad():

        if forecast_model is not None:

            pred = forecast_model(batch_x_dev, adj_dev).detach().cpu().numpy()[0, :, :, 0]

            forecast_vals = scaler.inverse_transform(pred)

        else:

            last_frame = batch_x.numpy()[0, -1, :, 0]

            forecast_vals = np.tile(scaler.inverse_transform(last_frame[None, :]), (CFG['forecast_seq_out'], 1))



    current_vals = scaler.inverse_transform(batch_x.numpy()[0, -1, :, 0][None, :])[0]

    future_mean = forecast_vals.mean(axis=0)

    trend = future_mean - current_vals



    num_pseudo_zones = CFG['zone_rows'] * CFG['zone_cols']

    sensor_groups = np.array_split(np.arange(len(current_vals)), num_pseudo_zones)

    rows = []

    for zone_idx, sensor_ids in enumerate(sensor_groups):

        rows.append({

            'zone': f'Zone-{chr(65 + zone_idx)}',

            'current_load': float(current_vals[sensor_ids].mean()),

            'forecast_load': float(future_mean[sensor_ids].mean()),

            'trend': float(trend[sensor_ids].mean()),

        })

    forecast_zone_df = pd.DataFrame(rows)

    forecast_zone_df['congestion_risk'] = softmax_np(np.maximum(forecast_zone_df['trend'].values, 0.0))



forecast_zone_df.sort_values('congestion_risk', ascending=False).reset_index(drop=True)

In [ ]:
# Visual anomaly evidence from the same UCSD clip

mid_frame = clip_gray[len(clip_gray) // 2]

anomaly_score = 0.0

anomaly_map = flow_map.copy()



if anomaly_loaded:

    frame_tensor = torch.tensor(mid_frame[None, None, :, :], dtype=torch.float32, device=DEVICE)

    with torch.no_grad():

        recon = anomaly_model(frame_tensor).detach().cpu().numpy()[0, 0]

    anomaly_map = np.abs(mid_frame - recon)

    anomaly_score = float(anomaly_map.mean())

else:

    anomaly_score = float(flow_map.mean())



anomaly_zones = []

for zone in zone_grid(anomaly_map.shape[0], anomaly_map.shape[1], CFG['zone_rows'], CFG['zone_cols']):

    patch = anomaly_map[zone['r0']:zone['r1'], zone['c0']:zone['c1']]

    anomaly_zones.append({'zone': zone['name'], 'anomaly_signal': float(patch.mean())})



anomaly_df = pd.DataFrame(anomaly_zones).sort_values('anomaly_signal', ascending=False).reset_index(drop=True)

anomaly_df

In [ ]:
# Fuse density, flow, anomaly, and forecast into zone-level event risk

risk_df = zone_density_df[['zone', 'density_sum', 'density_mean']].copy()

risk_df = risk_df.merge(flow_df[['zone', 'flow_intensity']], on='zone', how='left')

risk_df = risk_df.merge(anomaly_df[['zone', 'anomaly_signal']], on='zone', how='left')

if not forecast_zone_df.empty:

    risk_df = risk_df.merge(forecast_zone_df[['zone', 'trend', 'congestion_risk']], on='zone', how='left')

else:

    risk_df['trend'] = 0.0

    risk_df['congestion_risk'] = 0.0



for col in ['flow_intensity', 'anomaly_signal', 'trend', 'congestion_risk']:

    risk_df[col] = risk_df[col].fillna(0.0)



def minmax(col):

    x = risk_df[col].to_numpy(dtype=np.float64)

    return (x - x.min()) / (x.max() - x.min() + 1e-8)



risk_df['density_score'] = minmax('density_sum')

risk_df['flow_score'] = minmax('flow_intensity')

risk_df['anomaly_score'] = minmax('anomaly_signal')

risk_df['forecast_score'] = minmax('congestion_risk')



risk_df['risk_score'] = (

    0.35 * risk_df['density_score'] +

    0.20 * risk_df['flow_score'] +

    0.25 * risk_df['anomaly_score'] +

    0.20 * risk_df['forecast_score']

)



risk_df['risk_level'] = pd.cut(

    risk_df['risk_score'],

    bins=[-1, 0.25, 0.50, 0.75, 1.0],

    labels=['LOW', 'GUARDED', 'HIGH', 'CRITICAL']

)



risk_df = risk_df.sort_values('risk_score', ascending=False).reset_index(drop=True)

risk_df[['zone', 'risk_score', 'risk_level', 'density_sum', 'flow_intensity', 'anomaly_signal', 'congestion_risk']]

In [ ]:
# Build commander summary and dispatch suggestions

top_zone = risk_df.iloc[0]

top_three = risk_df.head(3)



def zone_anchor(zone_name):

    idx = ord(zone_name.split('-')[-1]) - ord('A')

    cols = CFG['zone_cols']

    row = idx // cols

    col = idx % cols

    return ((col + 0.5) / cols, (row + 0.5) / CFG['zone_rows'])



incidents = []

for _, row in top_three.iterrows():

    incidents.append({

        'zone': row['zone'],

        'risk_level': row['risk_level'],

        'risk_score': float(row['risk_score']),

        'x': zone_anchor(row['zone'])[0],

        'y': zone_anchor(row['zone'])[1],

        'reason': ', '.join([

            'high density' if row['density_score'] > 0.65 else 'moderate density',

            'strong flow' if row['flow_score'] > 0.65 else 'stable flow',

            'anomaly evidence' if row['anomaly_score'] > 0.65 else 'normal behavior',

            'forecasted congestion' if row['forecast_score'] > 0.65 else 'stable forecast',

        ])

    })



dispatch_rows = []

for incident in incidents:

    best_unit = None

    best_cost = float('inf')

    for unit in CFG['dispatch_units']:

        base_dist = math.dist((unit['x'], unit['y']), (incident['x'], incident['y']))

        congestion_penalty = 1.0 + incident['risk_score']

        route_cost = base_dist * congestion_penalty

        if route_cost < best_cost:

            best_cost = route_cost

            best_unit = unit

    dispatch_rows.append({

        'incident_zone': incident['zone'],

        'risk_level': incident['risk_level'],

        'recommended_unit': best_unit['unit_id'],

        'unit_type': best_unit['type'],

        'route_cost': round(best_cost, 4),

        'reason': incident['reason'],

    })



dispatch_df = pd.DataFrame(dispatch_rows)



summary_lines = [

    'CrowdVision Event Safety Brief',

    f"Highest-risk zone: {top_zone['zone']} ({top_zone['risk_level']}) with score {top_zone['risk_score']:.2f}.",

    f"Top density pressure observed in {zone_density_df.iloc[0]['zone']}.",

    f"Top motion anomaly observed in {anomaly_df.iloc[0]['zone']}.",

]

if not forecast_zone_df.empty:

    forecast_top = forecast_zone_df.sort_values('congestion_risk', ascending=False).iloc[0]

    summary_lines.append(

        f"Forecast indicates rising congestion in {forecast_top['zone']} with trend {forecast_top['trend']:.2f}."

    )

summary_lines.append('Dispatch priority:')

for _, row in dispatch_df.iterrows():

    summary_lines.append(

        f"- Send {row['recommended_unit']} to {row['incident_zone']} ({row['risk_level']}); trigger reason: {row['reason']}."

    )



commander_summary = '\n'.join(summary_lines)

print(commander_summary)

dispatch_df

In [ ]:
# Visual dashboard and exports

fig, axes = plt.subplots(2, 3, figsize=(18, 10))



axes[0, 0].imshow(img_np)

axes[0, 0].set_title('Crowd Image')

axes[0, 0].axis('off')



im1 = axes[0, 1].imshow(density_pred, cmap='jet')

axes[0, 1].set_title('Density Map')

axes[0, 1].axis('off')

plt.colorbar(im1, ax=axes[0, 1], fraction=0.046)



im2 = axes[0, 2].imshow(flow_map, cmap='magma')

axes[0, 2].set_title('Flow Intensity Proxy')

axes[0, 2].axis('off')

plt.colorbar(im2, ax=axes[0, 2], fraction=0.046)



im3 = axes[1, 0].imshow(anomaly_map, cmap='hot')

axes[1, 0].set_title('Anomaly Evidence Map')

axes[1, 0].axis('off')

plt.colorbar(im3, ax=axes[1, 0], fraction=0.046)



risk_plot = risk_df.sort_values('zone')

axes[1, 1].bar(risk_plot['zone'], risk_plot['risk_score'], color='tomato')

axes[1, 1].set_title('Zone Risk Scores')

axes[1, 1].set_ylim(0, 1.0)

axes[1, 1].tick_params(axis='x', rotation=45)

axes[1, 1].grid(True, axis='y', alpha=0.3)



dispatch_plot = dispatch_df.copy()

axes[1, 2].bar(dispatch_plot['incident_zone'], dispatch_plot['route_cost'], color='steelblue')

axes[1, 2].set_title('Dispatch Route Cost')

axes[1, 2].tick_params(axis='x', rotation=45)

axes[1, 2].grid(True, axis='y', alpha=0.3)



plt.tight_layout()

plt.savefig(EXPS_ROOT / 'event_safety_dashboard.png', dpi=150, bbox_inches='tight')

plt.show()



risk_df.to_csv(EXPS_ROOT / 'zone_risk_scores.csv', index=False)

dispatch_df.to_csv(EXPS_ROOT / 'dispatch_recommendations.csv', index=False)

(EXPS_ROOT / 'commander_summary.txt').write_text(commander_summary)



print('Saved: event_safety_dashboard.png, zone_risk_scores.csv, dispatch_recommendations.csv, commander_summary.txt')

---



## Event-Safety Intelligence Complete



This notebook produces the operational ML outputs needed by the rest of the team:



1. Zone-level risk scores

2. Dispatch recommendation table

3. Commander summary text

4. Visual dashboard export



Continue with **Notebook 05** for unified multi-task training.